# 双均线交叉策略 — 药明康德(603259.SH)

**策略逻辑**：MA5（短均线）上穿 MA15（长均线）→ 金叉买入；MA5 下穿 MA15 → 死叉卖出。

**内容**：
1. 加载前复权日线数据
2. 计算短均线(MA5)和长均线(MA15)
3. 计算金叉/死叉买卖信号
4. 可视化：股价 + 均线 + 买卖标记
5. 模拟回测：总收益/年化/夏普/最大回撤/胜率/盈亏比等

In [ ]:
import csv, os, json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties

# ── 配置 ──
CSV_PATH = r'G:\learn\test\data\603259_SH_daily.csv'
OUT_DIR  = r'G:\learn\test'
CHART_DIR = os.path.join(OUT_DIR, 'charts')
os.makedirs(CHART_DIR, exist_ok=True)

# 中文字体
SIMSUN = 'C:/Windows/Fonts/simsun.ttc'
mpl_font = FontProperties(fname=SIMSUN)
plt.rcParams['axes.unicode_minus'] = False

# 中国股市配色：涨红跌绿
COLOR_UP   = '#e74c3c'
COLOR_DOWN = '#27ae60'
COLOR_MA_S = '#e67e22'   # 短均线 橙
COLOR_MA_L = '#2980b9'   # 长均线 蓝

# 策略参数
SHORT_PERIOD = 5
LONG_PERIOD  = 15

# 回测参数
INITIAL_CAPITAL = 100000.0   # 初始资金 10万
FEE_RATE = 0.0008            # 手续费率 万八（含佣金+滑点）
STAMP_TAX = 0.0005           # 印花税 万五（卖出单边）
POSITION_SIZE = 0.95         # 仓位比例 95%

print('配置加载完成')

## Step 1: 加载已存储的股价数据

In [ ]:
def load_data(path):
    dates, close, vol, amount = [], [], [], []
    with open(path, 'r', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            dates.append(row['trade_date'])
            close.append(float(row['close_qfq']))
            vol.append(float(row['vol']))
            amount.append(float(row['amount']))
    return dates, np.array(close), np.array(vol), np.array(amount)

dates, close, vol, amount = load_data(CSV_PATH)
n = len(close)
print(f'数据行数: {n}')
print(f'日期范围: {dates[0]} ~ {dates[-1]}')
print(f'最新收盘价(前复权): {close[-1]:.2f}')

## Step 2: 设定短均线和长均线的周期，计算均线数据

In [ ]:
def calc_sma(arr, period):
    """简单移动平均"""
    out = np.full_like(arr, np.nan, dtype=float)
    for i in range(period - 1, len(arr)):
        out[i] = np.mean(arr[i - period + 1: i + 1])
    return out

ma_short = calc_sma(close, SHORT_PERIOD)
ma_long  = calc_sma(close, LONG_PERIOD)

print(f'短均线 MA{SHORT_PERIOD}: 最新值 {ma_short[-1]:.2f}')
print(f'长均线 MA{LONG_PERIOD}: 最新值 {ma_long[-1]:.2f}')
print(f'当前状态: {"多头排列(短>长)" if ma_short[-1] > ma_long[-1] else "空头排列(短<长)"}')

## Step 3: 计算买入卖出的交易信号

- **金叉**：短均线从下方穿越到上方 → 买入信号
- **死叉**：短均线从上方穿越到下方 → 卖出信号

In [ ]:
signals = []  # list of (index, type: 'buy'/'sell', price)
position = False  # 是否持仓

for i in range(LONG_PERIOD, n):
    if np.isnan(ma_short[i]) or np.isnan(ma_long[i]):
        continue
    prev_diff = ma_short[i-1] - ma_long[i-1]
    curr_diff = ma_short[i]   - ma_long[i]
    
    # 金叉: 前一天短<=长，今天短>长
    if prev_diff <= 0 and curr_diff > 0 and not position:
        signals.append((i, 'buy', close[i]))
        position = True
    # 死叉: 前一天短>=长，今天短<长
    elif prev_diff >= 0 and curr_diff < 0 and position:
        signals.append((i, 'sell', close[i]))
        position = False

# 如果最后还持仓，以最后收盘价平仓
if position:
    signals.append((n-1, 'sell', close[-1]))
    position = False

print(f'共 {len(signals)} 次交易信号（{len([s for s in signals if s[1]=="buy"])} 次买入 + {len([s for s in signals if s[1]=="sell"])} 次卖出）')
print()
for idx, sig_type, price in signals:
    print(f'  {dates[idx]}  {sig_type.upper():4s}  {price:.2f}')

## Step 4: 绘制可视化图形

包含：股价、短/长均线、买入信号标记（红▲）、卖出信号标记（绿▼）

In [ ]:
x = list(range(n))
xticks_idx = list(range(0, n, max(1, n // 12)))
xticks_labels = [dates[i] for i in xticks_idx]

fig, ax = plt.subplots(figsize=(16, 8))

# 股价
ax.plot(x, close, color='#2c3e50', linewidth=1.2, label='收盘价(前复权)', zorder=2)
# 短均线
ax.plot(x, ma_short, color=COLOR_MA_S, linewidth=1.2, label=f'MA{SHORT_PERIOD}(短均线)', zorder=3)
# 长均线
ax.plot(x, ma_long, color=COLOR_MA_L, linewidth=1.2, label=f'MA{LONG_PERIOD}(长均线)', zorder=3)

# 买卖标记
buy_idx   = [s[0] for s in signals if s[1] == 'buy']
buy_price = [s[2] for s in signals if s[1] == 'buy']
sell_idx   = [s[0] for s in signals if s[1] == 'sell']
sell_price = [s[2] for s in signals if s[1] == 'sell']

if buy_idx:
    ax.scatter(buy_idx, buy_price, color=COLOR_UP, marker='^', s=120,
              zorder=5, label=f'买入信号({len(buy_idx)}次)', edgecolors='white', linewidths=0.5)
if sell_idx:
    ax.scatter(sell_idx, sell_price, color=COLOR_DOWN, marker='v', s=120,
              zorder=5, label=f'卖出信号({len(sell_idx)}次)', edgecolors='white', linewidths=0.5)

# 金叉/死叉垂直线
for idx, sig_type, _ in signals:
    color = COLOR_UP if sig_type == 'buy' else COLOR_DOWN
    ax.axvline(x=idx, color=color, alpha=0.15, linewidth=0.8)

# 标注买卖文字
for idx, sig_type, price in signals:
    label = '买' if sig_type == 'buy' else '卖'
    offset = price * 0.03 if sig_type == 'buy' else -price * 0.03
    ax.annotate(label, xy=(idx, price), xytext=(idx, price + offset),
               fontproperties=mpl_font, fontsize=9, ha='center',
               color=COLOR_UP if sig_type == 'buy' else COLOR_DOWN, fontweight='bold')

ax.set_title(f'药明康德(603259.SH) 双均线交叉策略 (MA{SHORT_PERIOD}/MA{LONG_PERIOD})',
             fontproperties=mpl_font, fontsize=15)
ax.set_ylabel('价格(元)', fontproperties=mpl_font, fontsize=12)
ax.set_xlabel('交易日期', fontproperties=mpl_font, fontsize=12)
ax.legend(prop=mpl_font, fontsize=10, loc='upper left')
ax.set_xticks(xticks_idx)
ax.set_xticklabels(xticks_labels, rotation=45, ha='right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
chart_path = os.path.join(CHART_DIR, '603259_SH_ma_strategy.png')
fig.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'图表已保存: {chart_path}')

## Step 5: 模拟交易和回测，计算量化指标

In [ ]:
capital = INITIAL_CAPITAL
position_shares = 0
entry_price = 0
entry_idx = 0

trade_records = []   # 每笔完整交易记录
daily_values = []    # 每日账户净值

# 构建信号字典
signal_dict = {s[0]: s[1] for s in signals}

for i in range(n):
    # 每日净值
    if position_shares > 0:
        daily_value = capital + position_shares * close[i]
    else:
        daily_value = capital
    daily_values.append(daily_value)
    
    if i in signal_dict:
        sig = signal_dict[i]
        if sig == 'buy':
            buy_amount = capital * POSITION_SIZE
            shares = int(buy_amount / close[i] / 100) * 100  # 整手
            if shares == 0:
                shares = int(buy_amount / close[i])
            cost = shares * close[i] * (1 + FEE_RATE)
            capital -= cost
            position_shares = shares
            entry_price = close[i]
            entry_idx = i
            
        elif sig == 'sell':
            revenue = position_shares * close[i] * (1 - FEE_RATE - STAMP_TAX)
            capital += revenue
            pnl = revenue - position_shares * entry_price * (1 + FEE_RATE)
            pnl_pct = pnl / (position_shares * entry_price * (1 + FEE_RATE)) * 100
            hold_days = i - entry_idx
            trade_records.append({
                'buy_date': dates[entry_idx],
                'buy_price': entry_price,
                'sell_date': dates[i],
                'sell_price': close[i],
                'hold_days': hold_days,
                'pnl': pnl,
                'pnl_pct': pnl_pct,
                'win': 1 if pnl > 0 else 0
            })
            position_shares = 0

daily_values = np.array(daily_values)
print('回测模拟完成')

In [ ]:
# ── 量化指标计算 ──
final_value = daily_values[-1]
total_return = (final_value - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100

# 年化收益率
annual_return = ((final_value / INITIAL_CAPITAL) ** (252 / n) - 1) * 100

# 日收益率
daily_returns = np.diff(daily_values) / daily_values[:-1]

# 夏普比率 (无风险利率 3%)
rf_daily = 0.03 / 252
excess_returns = daily_returns - rf_daily
sharpe = np.mean(excess_returns) / np.std(excess_returns) * np.sqrt(252) if np.std(excess_returns) > 0 else 0

# 最大回撤
peak = np.maximum.accumulate(daily_values)
drawdown = (daily_values - peak) / peak
max_drawdown = np.min(drawdown) * 100
max_dd_idx = np.argmin(drawdown)
peak_idx = np.argmax(daily_values[:max_dd_idx+1]) if max_dd_idx > 0 else 0

# 交易统计
n_trades = len(trade_records)
n_wins = sum(t['win'] for t in trade_records)
n_losses = n_trades - n_wins
win_rate = n_wins / n_trades * 100 if n_trades > 0 else 0
avg_hold = np.mean([t['hold_days'] for t in trade_records]) if trade_records else 0
avg_win_pct = np.mean([t['pnl_pct'] for t in trade_records if t['win']]) if n_wins > 0 else 0
avg_loss_pct = np.mean([t['pnl_pct'] for t in trade_records if not t['win']]) if n_losses > 0 else 0

# 盈亏比
total_profit = sum(t['pnl'] for t in trade_records if t['win'])
total_loss = abs(sum(t['pnl'] for t in trade_records if not t['win']))
profit_factor = total_profit / total_loss if total_loss > 0 else float('inf')

# 买入持有对比
bh_return = (close[-1] - close[0]) / close[0] * 100
bh_value = INITIAL_CAPITAL * (close[-1] / close[0])

print('=' * 60)
print(f'  双均线交叉策略回测报告 — 药明康德(603259.SH)')
print(f'  策略参数: MA{SHORT_PERIOD} / MA{LONG_PERIOD}')
print(f'  回测区间: {dates[0]} ~ {dates[-1]} ({n} 个交易日)')
print('=' * 60)

print(f'\n【收益指标】')
print(f'  初始资金:       ¥{INITIAL_CAPITAL:,.0f}')
print(f'  最终净值:       ¥{final_value:,.2f}')
print(f'  策略总收益:     {total_return:+.2f}%')
print(f'  年化收益率:     {annual_return:+.2f}%')
print(f'  买入持有收益:   {bh_return:+.2f}%')
print(f'  买入持有净值:   ¥{bh_value:,.2f}')
print(f'  超额收益:       {total_return - bh_return:+.2f}%')

print(f'\n【风险指标】')
print(f'  夏普比率:       {sharpe:.2f}')
print(f'  最大回撤:       {max_drawdown:.2f}%')
print(f'  回撤高点:       {dates[peak_idx]}')
print(f'  回撤低点:       {dates[max_dd_idx]}')

print(f'\n【交易统计】')
print(f'  总交易次数:     {n_trades}')
print(f'  盈利次数:       {n_wins}')
print(f'  亏损次数:       {n_losses}')
print(f'  胜率:           {win_rate:.1f}%')
print(f'  平均持仓天数:   {avg_hold:.0f}')
print(f'  平均盈利幅度:   {avg_win_pct:+.2f}%')
print(f'  平均亏损幅度:   {avg_loss_pct:+.2f}%')
print(f'  盈亏比:         {profit_factor:.2f}')

In [ ]:
# 逐笔交易明细
print(f'\n【逐笔交易明细】')
print(f'  {"序号":>3} {"买入日期":>10} {"买入价":>8} {"卖出日期":>10} {"卖出价":>8} {"持仓天":>5} {"收益额":>10} {"收益率":>8}')
print(f'  {"─"*3} {"─"*10} {"─"*8} {"─"*10} {"─"*8} {"─"*5} {"─"*10} {"─"*8}')
for i, t in enumerate(trade_records):
    print(f'  {i+1:>3} {t["buy_date"]:>10} {t["buy_price"]:>8.2f} {t["sell_date"]:>10} {t["sell_price"]:>8.2f} '
          f'{t["hold_days"]:>5} ¥{t["pnl"]:>9,.2f} {t["pnl_pct"]:>+7.2f}%')

In [ ]:
# 净值曲线 + 回撤图
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

# 净值曲线
ax1.plot(x, daily_values, color='#2c3e50', linewidth=1.2, label='策略净值')
ax1.plot(x, np.full(n, INITIAL_CAPITAL), color='gray', linewidth=0.8, linestyle='--', label='初始资金')
# 买入持有对比
bh_curve = INITIAL_CAPITAL * close / close[0]
ax1.plot(x, bh_curve, color='#f39c12', linewidth=1, linestyle=':', label='买入持有')
ax1.set_title('药明康德(603259.SH) 双均线策略净值曲线 vs 买入持有',
              fontproperties=mpl_font, fontsize=14)
ax1.set_ylabel('净值(元)', fontproperties=mpl_font)
ax1.legend(prop=mpl_font, fontsize=10)
ax1.grid(True, alpha=0.3)

# 回撤曲线
ax2.fill_between(x, drawdown * 100, 0, color=COLOR_DOWN, alpha=0.3)
ax2.plot(x, drawdown * 100, color=COLOR_DOWN, linewidth=0.8)
ax2.set_title('回撤曲线', fontproperties=mpl_font, fontsize=12)
ax2.set_ylabel('回撤(%)', fontproperties=mpl_font)
ax2.set_xlabel('交易日期', fontproperties=mpl_font)
ax2.grid(True, alpha=0.3)

for ax in [ax1, ax2]:
    ax.set_xticks(xticks_idx)
    ax.set_xticklabels(xticks_labels, rotation=45, ha='right')

plt.tight_layout()
nav_path = os.path.join(CHART_DIR, '603259_SH_nav_drawdown.png')
fig.savefig(nav_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'净值回撤图已保存: {nav_path}')

In [ ]:
# 保存回测结果 JSON
result = {
    'stock': '药明康德(603259.SH)',
    'strategy': f'双均线交叉 MA{SHORT_PERIOD}/MA{LONG_PERIOD}',
    'period': f'{dates[0]} ~ {dates[-1]}',
    'trading_days': n,
    'initial_capital': INITIAL_CAPITAL,
    'final_value': round(final_value, 2),
    'total_return_pct': round(total_return, 2),
    'annual_return_pct': round(annual_return, 2),
    'buy_hold_return_pct': round(bh_return, 2),
    'excess_return_pct': round(total_return - bh_return, 2),
    'sharpe_ratio': round(sharpe, 2),
    'max_drawdown_pct': round(max_drawdown, 2),
    'n_trades': n_trades,
    'n_wins': n_wins,
    'n_losses': n_losses,
    'win_rate_pct': round(win_rate, 1),
    'avg_hold_days': round(avg_hold, 0),
    'avg_win_pct': round(avg_win_pct, 2),
    'avg_loss_pct': round(avg_loss_pct, 2),
    'profit_factor': round(profit_factor, 2),
    'trades': trade_records
}

json_path = os.path.join(OUT_DIR, 'backtest_result.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print(f'回测结果已保存: {json_path}')
print('\n=== 回测完成 ===')